# Lesson 8: Tree-Based Models

A completely different style of model from the line-fitting we've done. **No weights, no gradient descent** — trees learn by asking a series of yes/no questions.

We'll build up:
1. **Decision Tree** — one tree of questions.
2. **Random Forest** — many trees voting together.
3. **Gradient Boosting / XGBoost** — trees that fix each other's mistakes.

These dominate real-world tabular (spreadsheet) data and Kaggle competitions.

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Part 1: Decision Tree — a flowchart of yes/no questions

A decision tree is literally a flowchart it builds from the data:

```
                 [hours studied > 5?]
                 /                  \
               yes                   no
                |                     |
       [attendance > 80%?]          FAIL
        /            \
      yes             no
       |               |
      PASS            FAIL
```

**How it picks the questions:** at each step it chooses the split that best separates the classes (using a 'purity' measure like Gini impurity or entropy). It keeps splitting until groups are pure or it hits a depth limit.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(f'Decision Tree test accuracy: {accuracy_score(y_test, tree.predict(X_test)):.3f}')

### Visualize the tree (this is why trees are loved — you can READ them)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 8))
plot_tree(tree, feature_names=data.feature_names, class_names=data.target_names,
          filled=True, fontsize=8)
plt.show()

### The weakness: a single deep tree OVERFITS

Let a tree grow unlimited and it memorizes the training data (Lesson 7!). Watch train accuracy hit ~100% while test accuracy lags.

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=42)  # no depth limit
deep_tree.fit(X_train, y_train)
print(f'Train accuracy: {accuracy_score(y_train, deep_tree.predict(X_train)):.3f}  <- memorized!')
print(f'Test accuracy:  {accuracy_score(y_test, deep_tree.predict(X_test)):.3f}')

## Part 2: Random Forest — wisdom of the crowd

One tree overfits. So build **many** trees, each on a random slice of data + random subset of features, and let them **vote**. Individual mistakes cancel out; the majority is robust.

This is called an **ensemble** (many models combined). Random Forest is one of the most reliable off-the-shelf models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
print(f'Random Forest test accuracy: {accuracy_score(y_test, forest.predict(X_test)):.3f}')
print('(n_estimators=100 means 100 trees voting)')

### Bonus: trees tell you which features matter

A great practical perk — feature importance for free.

In [ ]:
importances = forest.feature_importances_
top = np.argsort(importances)[-8:][::-1]
print('Top 8 most important features:')
for i in top:
    print(f'  {data.feature_names[i]:<25} {importances[i]:.3f}')

## Part 3: Gradient Boosting — trees that fix each other

Different idea from a forest. Instead of independent trees voting, boosting builds trees **one after another**, where **each new tree focuses on the errors the previous trees made**. Slow, sequential, but extremely accurate.

**XGBoost / LightGBM** are famous optimized versions — they win a huge share of Kaggle tabular competitions. Below we use sklearn's built-in version.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
print(f'Gradient Boosting test accuracy: {accuracy_score(y_test, gb.predict(X_test)):.3f}')

## Summary comparison

In [ ]:
models = {
    'Decision Tree (depth 3)': tree,
    'Decision Tree (deep)':    deep_tree,
    'Random Forest':           forest,
    'Gradient Boosting':       gb,
}
print(f"{'Model':<26} {'Test accuracy':>14}")
print('-' * 42)
for name, m in models.items():
    print(f'{name:<26} {accuracy_score(y_test, m.predict(X_test)):>13.3f}')

## Key takeaways

| Model | How it works | Strength | Weakness |
|---|---|---|---|
| Decision Tree | one flowchart of yes/no splits | readable, interpretable | overfits easily |
| Random Forest | many trees vote (bagging) | robust, little tuning | less interpretable |
| Gradient Boosting / XGBoost | trees fix prior trees' errors | top accuracy on tabular | needs tuning, slower |

- Trees use **no weights and no gradient descent** — they split on questions.
- Trees **don't need feature scaling** (unlike linear models / PCA / K-Means).
- For most spreadsheet/tabular problems, Random Forest or XGBoost are strong default choices.

## Your turn

1. Change `max_depth` of the first tree to 1, 2, 5. How does test accuracy change?
2. Change Random Forest `n_estimators` to 5 vs 500. Does more trees always help?
3. In your own words: how is a Random Forest different from Gradient Boosting?
4. Why does a single deep decision tree overfit, but a forest of them doesn't?